# Statistical Analysis: Marketing Channel Performance

This notebook contains the full workflow for the project:
- data exploration and cleaning
- daily aggregation by marketing platform
- pairwise statistical testing on CPA and conversion rates
- confidence intervals, multiple-comparison corrections, and power analysis

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, fisher_exact, false_discovery_control

SCRIPT_DIR = os.path.dirname(os.path.abspath("__file__"))
sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Data Exploration and Preparation

Load the raw marketing dataset, clean it, aggregate it to daily platform level, and save the prepared CSV for the statistical analysis that follows.


In [ ]:
HF_URL = (
    "hf://datasets/jason1966/alinaboulsi_digital-marketing-performance-dataset"
    "/digital_marketing_dataset_30k.csv"
)
print("Loading dataset from HuggingFace ...")
df = pd.read_csv(HF_URL)

print("\n=== Dataset Overview ===")
print(f"Shape:   {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("\nDtypes:")
print(df.dtypes.to_string())
print("\nMissing values:")
missing = df.isnull().sum()
print(missing[missing > 0].to_string() if (missing > 0).any() else "None")

NUM_COLS = ["spend", "impressions", "clicks", "conversions", "revenue", "reach", "video_views"]
df[NUM_COLS] = df[NUM_COLS].fillna(0)
df = df[(df["spend"] > 0) | (df["impressions"] > 0)].copy()
df["date"] = pd.to_datetime(df["date"], dayfirst=True)

print(f"\nAfter cleaning: {df.shape[0]:,} rows remain")

GROUP_COL = "platform"
daily = df.groupby([GROUP_COL, "date"])[NUM_COLS].sum().reset_index()

def safe_div(a, b):
    return np.where(b > 0, a / b, np.nan)

daily["CTR"] = safe_div(daily["clicks"], daily["impressions"])
daily["conversion_rate"] = safe_div(daily["conversions"], daily["clicks"])
daily["CPA"] = safe_div(daily["spend"], daily["conversions"])
daily["ROAS"] = safe_div(daily["revenue"], daily["spend"])
daily["profit"] = daily["revenue"] - daily["spend"]
daily["profit_margin"] = safe_div(daily["profit"], daily["revenue"])
daily.replace([np.inf, -np.inf], np.nan, inplace=True)

platforms = sorted(daily["platform"].unique())
print(f"\nDaily platform observations: {len(daily):,}")
print(daily.groupby("platform").size().rename("n_days").to_string())

groups = df.groupby(GROUP_COL)[NUM_COLS].sum().reset_index()
groups["CTR"] = safe_div(groups["clicks"], groups["impressions"])
groups["conversion_rate"] = safe_div(groups["conversions"], groups["clicks"])
groups["CPA"] = safe_div(groups["spend"], groups["conversions"])
groups["ROAS"] = safe_div(groups["revenue"], groups["spend"])
groups["profit"] = groups["revenue"] - groups["spend"]
groups["profit_margin"] = safe_div(groups["profit"], groups["revenue"])
groups.replace([np.inf, -np.inf], np.nan, inplace=True)
groups = groups.round(4)

DISPLAY_COLS = ["platform", "spend", "impressions", "clicks", "conversions", "revenue", "CTR", "conversion_rate", "CPA", "ROAS", "profit", "profit_margin"]
print("\n=== Aggregated Marketing Metrics by Platform ===")
print(groups[DISPLAY_COLS].to_string(index=False))

metrics_spec = [
    ("CPA", "CPA by Platform  (lower = better)", False),
    ("ROAS", "ROAS by Platform  (higher = better)", False),
    ("conversion_rate", "Conversion Rate by Platform", False),
    ("conversions", "Total Conversions by Platform", False),
    ("spend", "Total Spend ($) by Platform", False),
    ("profit", "Profit ($) by Platform", True),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
for ax, (metric, title, zero_line) in zip(axes, metrics_spec):
    data = groups[["platform", metric]].dropna().sort_values(metric)
    bars = ax.barh(data["platform"], data[metric], color="steelblue", edgecolor="white")
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel(metric)
    if zero_line:
        ax.axvline(0, color="red", linestyle="--", linewidth=1.2)
    for bar in bars:
        w = bar.get_width()
        ax.text(w * 1.01 if w >= 0 else w * 0.99, bar.get_y() + bar.get_height() / 2, f"{w:,.2f}", va="center", fontsize=8)
plt.suptitle("Marketing Channel Performance Overview", fontsize=14, fontweight="bold")
plt.tight_layout()
out = os.path.join(SCRIPT_DIR, "group_metrics_overview.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {out}")

DIST_METRICS = ["CPA", "ROAS", "conversion_rate"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, DIST_METRICS):
    for platform in sorted(daily["platform"].unique()):
        vals = daily.loc[daily["platform"] == platform, metric].dropna()
        ax.hist(vals, bins=30, alpha=0.45, label=platform, edgecolor="none")
    ax.set_title(f"Daily {metric} distribution by platform")
    ax.set_xlabel(metric)
    ax.legend(fontsize=7)
plt.suptitle("Daily Metric Distributions by Marketing Platform", fontsize=13)
plt.tight_layout()
out = os.path.join(SCRIPT_DIR, "group_distributions.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

platforms_sorted = sorted(daily["platform"].unique())
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, DIST_METRICS):
    data_per = [daily.loc[daily["platform"] == p, metric].dropna().values for p in platforms_sorted]
    bp = ax.boxplot(data_per, labels=platforms_sorted, patch_artist=True, medianprops={"color": "black", "linewidth": 1.5})
    colors = sns.color_palette("muted", len(platforms_sorted))
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(f"{metric} boxplot by Platform")
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Per-day Metric Variability by Marketing Platform", fontsize=13)
plt.tight_layout()
out = os.path.join(SCRIPT_DIR, "group_distributions_boxplot.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

out_csv = os.path.join(SCRIPT_DIR, "marketing_data.csv")
daily.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}  (shape: {daily.shape})")

print("\n=== Final Summary ===")
print(f"Platforms analysed: {platforms_sorted}")
for p in platforms_sorted:
    n = (daily["platform"] == p).sum()
    sub = daily[daily["platform"] == p]
    print(f"  {p:<18}  {n:>4} daily obs  |  median CPA={sub['CPA'].median():.2f}  median ROAS={sub['ROAS'].median():.2f}  conv_rate={sub['conversion_rate'].median():.4f}")

## 2. Helper Functions

**Cohen's d** measures practical significance — how many pooled standard deviations apart the two group means are.

| |d| range | Interpretation |
|---|---|
| < 0.2 | negligible |
| 0.2 – 0.5 | small |
| 0.5 – 0.8 | medium |
| ≥ 0.8 | large |

In [ ]:
def cohens_d(a, b):
    pooled_std = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2)
    return (np.mean(b) - np.mean(a)) / pooled_std if pooled_std > 0 else 0.0

def effect_label(d):
    d = abs(d)
    if d < 0.2:  return "negligible"
    if d < 0.5:  return "small"
    if d < 0.8:  return "medium"
    return "large"

## 3. Pairwise t-Tests on Daily CPA

An independent two-sample t-test checks whether the **mean daily Cost Per Acquisition** differs significantly between each pair of platforms.

$$t = \frac{\bar{x}_B - \bar{x}_A}{\text{SE}_{\text{diff}}}$$

We require at least 3 observations per group before running the test.

In [ ]:
cpa_clean = daily[["platform", "CPA"]].dropna()
cpa_clean = cpa_clean[np.isfinite(cpa_clean["CPA"])]

cpa_results = []
for i, a in enumerate(platforms):
    for b in platforms[i + 1:]:
        vals_a = cpa_clean.loc[cpa_clean["platform"] == a, "CPA"].values
        vals_b = cpa_clean.loc[cpa_clean["platform"] == b, "CPA"].values
        if len(vals_a) < 3 or len(vals_b) < 3:
            continue
        t_stat, p_val = ttest_ind(vals_a, vals_b)
        diff     = np.mean(vals_b) - np.mean(vals_a)
        pct_diff = (diff / np.mean(vals_a)) * 100 if np.mean(vals_a) != 0 else np.nan
        d = cohens_d(vals_a, vals_b)
        cpa_results.append({
            "group_A":      a,
            "group_B":      b,
            "mean_A":       round(np.mean(vals_a), 2),
            "mean_B":       round(np.mean(vals_b), 2),
            "n_A":          len(vals_a),
            "n_B":          len(vals_b),
            "difference":   round(diff, 2),
            "pct_diff":     round(pct_diff, 1),
            "t_stat":       round(t_stat, 3),
            "p_value":      round(p_val, 4),
            "cohens_d":     round(d, 3),
            "effect_size":  effect_label(d),
            "significant":  p_val < 0.05,
        })

cpa_df = pd.DataFrame(cpa_results)
print("=== CPA Pairwise t-test Results ===")
display(cpa_df)
print(f"\nTotal CPA comparisons: {len(cpa_df)}")
print(f"Significant (p<0.05):  {cpa_df['significant'].sum()}")

### 3a. P-Value Heatmap

The heatmap shows the raw p-values for each platform pair. Red cells indicate statistically significant differences (p < 0.05); green cells indicate no significant difference.

In [ ]:
pval_matrix = pd.DataFrame(1.0, index=platforms, columns=platforms)
for _, row in cpa_df.iterrows():
    pval_matrix.loc[row["group_A"], row["group_B"]] = row["p_value"]
    pval_matrix.loc[row["group_B"], row["group_A"]] = row["p_value"]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pval_matrix.values, cmap="RdYlGn_r", vmin=0, vmax=0.10)
plt.colorbar(im, ax=ax, label="p-value")
ax.set_xticks(range(len(platforms))); ax.set_yticks(range(len(platforms)))
ax.set_xticklabels(platforms, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(platforms, fontsize=9)
for i in range(len(platforms)):
    for j in range(len(platforms)):
        ax.text(j, i, f"{pval_matrix.iloc[i, j]:.3f}",
                ha="center", va="center", fontsize=8)
ax.set_title("CPA Pairwise Comparison — p-values\n(red = statistically significant at α=0.05)",
             fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SCRIPT_DIR, "metric_comparison_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: metric_comparison_heatmap.png")

## 4. Fisher's Exact Test on Conversion Rates

Conversion outcomes are **binary** (converted / did not convert), so Fisher's exact test is more appropriate than a t-test. It tests whether the odds of conversion differ significantly between platforms using the full 2×2 contingency table.

| | Converted | Not converted |
|---|---|---|
| Platform A | conv_A | clicks_A − conv_A |
| Platform B | conv_B | clicks_B − conv_B |

In [ ]:
agg = (
    daily.groupby("platform")[["conversions", "clicks"]]
    .sum()
    .reset_index()
)
agg["non_conversions"] = agg["clicks"] - agg["conversions"]
agg["rate"]            = agg["conversions"] / agg["clicks"]

print("=== Aggregated Conversion Rate Summary ===")
display(agg[["platform", "conversions", "clicks", "non_conversions", "rate"]])

In [ ]:
fisher_results = []
for i, a in enumerate(platforms):
    for b in platforms[i + 1:]:
        row_a = agg[agg["platform"] == a].iloc[0]
        row_b = agg[agg["platform"] == b].iloc[0]
        table = [[int(row_a["conversions"]), int(row_a["non_conversions"])],
                 [int(row_b["conversions"]), int(row_b["non_conversions"])]]
        odds_ratio, p_val = fisher_exact(table, alternative="two-sided")
        diff = row_b["rate"] - row_a["rate"]
        fisher_results.append({
            "group_A":      a,
            "group_B":      b,
            "conv_A":       int(row_a["conversions"]),
            "clicks_A":     int(row_a["clicks"]),
            "rate_A":       round(row_a["rate"], 6),
            "conv_B":       int(row_b["conversions"]),
            "clicks_B":     int(row_b["clicks"]),
            "rate_B":       round(row_b["rate"], 6),
            "difference":   round(diff, 6),
            "odds_ratio":   round(odds_ratio, 4),
            "p_value":      p_val,
            "significant":  p_val < 0.05,
        })

fisher_df = pd.DataFrame(fisher_results)
print("=== Fisher's Exact Test Results (Conversion Rate) ===")
pd.set_option("display.float_format", lambda x: f"{x:.6e}" if abs(x) < 1e-3 else f"{x:.4f}")
display(fisher_df)
pd.reset_option("display.float_format")
print(f"\nTotal Fisher comparisons: {len(fisher_df)}")
print(f"Significant (p<0.05):     {fisher_df['significant'].sum()}")

### 4a. Conversion Rate Bar Chart

In [ ]:
agg_sorted = agg.sort_values("rate")
fig, ax = plt.subplots(figsize=(9, 5))
colors = sns.color_palette("muted", len(agg_sorted))
bars = ax.barh(agg_sorted["platform"], agg_sorted["rate"] * 100,
               color=colors, edgecolor="white")
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.05, bar.get_y() + bar.get_height() / 2,
            f"{w:.2f}%", va="center", fontsize=9)
ax.set_xlabel("Conversion Rate (%)")
ax.set_title("Conversion Rate by Marketing Platform", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SCRIPT_DIR, "rate_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rate_comparison.png")

## 5. Multiple Comparisons Correction

Running many tests simultaneously inflates the risk of false positives. With α=0.05 and many comparisons, we expect `n_tests × 0.05` spurious significant results by chance.

| Method | What it controls | Trade-off |
|---|---|---|
| **Bonferroni** | Family-wise error rate (FWER) | Very conservative — divides α by the number of tests |
| **Benjamini-Hochberg (FDR)** | Expected proportion of false discoveries | Less conservative — better power when many tests are run |

In [ ]:
alpha = 0.05
n_cpa    = len(cpa_df)
n_fisher = len(fisher_df)
n_total  = n_cpa + n_fisher

print("=== Multiple Comparisons Problem ===")
print(f"CPA t-tests:           {n_cpa}")
print(f"Fisher's exact tests:  {n_fisher}")
print(f"Total comparisons:     {n_total}")
print(f"Expected false positives at alpha=0.05: {n_total * alpha:.1f}")

# Bonferroni
alpha_bonf_cpa    = alpha / n_cpa
alpha_bonf_fisher = alpha / n_fisher
cpa_df["significant_bonferroni"]    = cpa_df["p_value"] < alpha_bonf_cpa
fisher_df["significant_bonferroni"] = fisher_df["p_value"] < alpha_bonf_fisher

# BH FDR
cpa_df["p_value_fdr"]       = false_discovery_control(cpa_df["p_value"].values, method="bh")
fisher_df["p_value_fdr"]    = false_discovery_control(fisher_df["p_value"].values, method="bh")
cpa_df["significant_fdr"]    = cpa_df["p_value_fdr"] < alpha
fisher_df["significant_fdr"] = fisher_df["p_value_fdr"] < alpha

print("\n=== Correction Summary ===")
for label, df_sub in [("CPA (t-test)", cpa_df), ("Conversion Rate (Fisher)", fisher_df)]:
    print(f"\n  {label}:")
    print(f"    Uncorrected (p<0.05):      {df_sub['significant'].sum()} / {len(df_sub)}")
    print(f"    Bonferroni corrected:       {df_sub['significant_bonferroni'].sum()} / {len(df_sub)}")
    print(f"    BH FDR (adj-p<0.05):        {df_sub['significant_fdr'].sum()} / {len(df_sub)}")

In [ ]:
if cpa_df["significant_fdr"].sum() > 0:
    print("FDR-significant CPA pairs:")
    sig = cpa_df[cpa_df["significant_fdr"]][
        ["group_A", "group_B", "mean_A", "mean_B", "pct_diff", "p_value", "p_value_fdr", "cohens_d"]]
    display(sig)

if fisher_df["significant_fdr"].sum() > 0:
    print("\nFDR-significant Conversion Rate pairs:")
    sig = fisher_df[fisher_df["significant_fdr"]][
        ["group_A", "group_B", "rate_A", "rate_B", "difference", "p_value", "p_value_fdr"]]
    display(sig)

### 5a. Correction Comparison Chart

How many significant pairs survive each correction method?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (cat, df_sub) in zip(axes, [("CPA (t-test)", cpa_df), ("Conversion Rate (Fisher)", fisher_df)]):
    counts = [
        df_sub["significant"].sum(),
        df_sub["significant_bonferroni"].sum(),
        df_sub["significant_fdr"].sum(),
    ]
    bars = ax.bar(["Uncorrected", "Bonferroni", "BH FDR"], counts,
                  color=["steelblue", "orange", "green"], edgecolor="white")
    ax.set_title(f"Significant {cat} Pairs\nby Correction Method", fontweight="bold")
    ax.set_ylabel("# Significant Pairs")
    ax.set_ylim(0, max(counts) + 2 if max(counts) > 0 else 5)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                str(count), ha="center", fontsize=11, fontweight="bold")

plt.suptitle("Effect of Multiple Comparisons Correction on Significant Results", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(SCRIPT_DIR, "correction_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: correction_comparison.png")

## 6. Bootstrap 95% Confidence Intervals for Daily CPA

Bootstrap resampling provides confidence intervals **without assuming a parametric distribution**. We resample daily CPA values with replacement 1 000 times and take the 2.5th and 97.5th percentiles of the bootstrap distribution of means.

A **narrower CI** means more consistent daily performance; a **wider CI** means higher day-to-day variability.

In [ ]:
def bootstrap_ci(data, n_bootstrap=1000, ci_level=0.95):
    rng = np.random.default_rng(42)
    means = [rng.choice(data, size=len(data), replace=True).mean()
             for _ in range(n_bootstrap)]
    lo = (1 - ci_level) / 2 * 100
    hi = (1 + ci_level) / 2 * 100
    return np.percentile(means, lo), np.percentile(means, hi)

print("=== 95% Bootstrap Confidence Intervals for Daily CPA by Platform ===")
ci_rows = []
for p in platforms:
    vals = cpa_clean.loc[cpa_clean["platform"] == p, "CPA"].values
    mean_cpa = np.mean(vals)
    lo, hi = bootstrap_ci(vals)
    ci_rows.append({"platform": p, "mean_CPA": round(mean_cpa, 2),
                    "CI_lower": round(lo, 2), "CI_upper": round(hi, 2)})
    print(f"  {p:<18}  mean=${mean_cpa:>7.2f}  95% CI=[{lo:.2f}, {hi:.2f}]")

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(os.path.join(SCRIPT_DIR, "cpa_confidence_intervals.csv"), index=False)
print("\nSaved: cpa_confidence_intervals.csv")

## 7. Save Results

In [ ]:
cpa_df.to_csv(os.path.join(SCRIPT_DIR, "cpa_comparison_results.csv"), index=False)
fisher_df.to_csv(os.path.join(SCRIPT_DIR, "fisher_comparison_results.csv"), index=False)
print("Saved: cpa_comparison_results.csv, fisher_comparison_results.csv")

## 8. Power Analysis

This section estimates how much data is needed to reliably detect CPA differences between marketing channels.

In [ ]:
rng = np.random.default_rng(42)

# Base CPA: median of all valid daily CPA values across all platforms
base_cpa = daily["CPA"].replace([np.inf, -np.inf], np.nan).dropna().median()
print(f"Base CPA for simulation: ${base_cpa:.2f}")

# Per-platform baseline CPAs
platform_cpa = (
    daily.groupby("platform")["CPA"]
    .median()
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)
print("\nPer-platform median daily CPA:")
print(platform_cpa.round(2).to_string())

def empirical_power_cpa(true_diff_pct, base_cpa, n_days, n_sim=1000, alpha=0.05):
    """Return the fraction of simulated experiments that reject H0."""
    std = base_cpa * 0.15
    cap = base_cpa * 0.50
    rejections = 0
    mean_b = base_cpa * (1 + true_diff_pct)
    for _ in range(n_sim):
        a = np.clip(rng.normal(base_cpa, std, n_days), base_cpa - cap, base_cpa + cap)
        b = np.clip(rng.normal(mean_b, std, n_days), mean_b - cap, mean_b + cap)
        _, p = ttest_ind(a, b)
        if p < alpha:
            rejections += 1
    return rejections / n_sim

EFFECT_SIZES = [0.05, 0.10, 0.15, 0.20]
SAMPLE_SIZES = [30, 60, 90, 120, 180]
N_SIM = 500

print(f"\nCalculating power grid ({N_SIM} simulations per cell) ...")
power_records = []
for eff in EFFECT_SIZES:
    for n in SAMPLE_SIZES:
        pw = empirical_power_cpa(eff, base_cpa, n, n_sim=N_SIM)
        power_records.append({
            "effect_size_pct": int(eff * 100),
            "n_days": n,
            "power": round(pw, 3),
        })
        print(f"  {int(eff * 100):>3}% diff  {n:>3} days  ->  power = {pw:.3f}")

power_df = pd.DataFrame(power_records)

colors = ["steelblue", "darkorange", "green", "crimson"]
fig, ax = plt.subplots(figsize=(9, 5))
for color, eff in zip(colors, EFFECT_SIZES):
    sub = power_df[power_df["effect_size_pct"] == int(eff * 100)]
    ax.plot(sub["n_days"], sub["power"], marker="o", color=color, linewidth=2,
            label=f"{int(eff * 100)}% CPA difference")
ax.axhline(0.80, color="black", linestyle="--", linewidth=1.5, label="80% power target")
ax.set_xlabel("Days of data  (sample size per channel)")
ax.set_ylabel("Statistical Power")
ax.set_title(f"Power to Detect CPA Differences\n(base CPA = ${base_cpa:.0f}, σ = 15%)",
             fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
power_plot_path = os.path.join(SCRIPT_DIR, "power_analysis_cpa.png")
plt.savefig(power_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {power_plot_path}")

pivot = power_df.pivot(index="n_days", columns="effect_size_pct", values="power")
pivot.columns = [f"{c}% diff" for c in pivot.columns]
print("\n=== Power Table (rows = days, cols = % CPA difference) ===")
print(pivot.round(3).to_string())

TARGET_POWER = 0.80
print("\n=== Minimum Days Required for 80% Power ===")
for eff in EFFECT_SIZES:
    sub = power_df[power_df["effect_size_pct"] == int(eff * 100)]
    sufficient = sub[sub["power"] >= TARGET_POWER]
    if len(sufficient) > 0:
        min_days = int(sufficient["n_days"].min())
        status = "SUFFICIENT (<=90 days)" if min_days <= 90 else f"NEED {min_days} days"
    else:
        min_days = ">180"
        status = "INSUFFICIENT at any tested N"
    print(f"  {int(eff * 100):>3}% difference  ->  min {min_days} days  [{status}]")

if "significant_fdr" in cpa_df.columns:
    sig_pairs = cpa_df[cpa_df["significant_fdr"] == True]
    if len(sig_pairs) > 0:
        print("\n=== Power Assessment for FDR-Significant CPA Pairs (90-day baseline) ===")
        for _, row in sig_pairs.iterrows():
            obs_pct = abs(row["pct_diff"]) / 100
            base_pair = min(row["mean_A"], row["mean_B"])
            pw_90 = empirical_power_cpa(obs_pct, base_pair, n_days=90, n_sim=N_SIM)
            min_n = next(
                (n for n in SAMPLE_SIZES
                 if empirical_power_cpa(obs_pct, base_pair, n, n_sim=300) >= TARGET_POWER),
                ">180",
            )
            adequate = "Adequate" if (isinstance(min_n, int) and min_n <= 90) else "Need more data"
            print(f"  {row['group_A']} vs {row['group_B']}: {abs(row['pct_diff']):.1f}% diff  "
                  f"90-day power={pw_90:.2f}  min_days={min_n}  [{adequate}]")
    else:
        print("\nNo FDR-significant CPA pairs to assess.")
else:
    print("\ncpa_comparison_results.csv does not contain 'significant_fdr' column — run the statistical section first.")

power_results_path = os.path.join(SCRIPT_DIR, "power_analysis_results.csv")
power_df.to_csv(power_results_path, index=False)
print(f"\nSaved: {power_results_path}")


## Summary

| Output file | Contents |
|---|---|
| `metric_comparison_heatmap.png` | Pairwise CPA p-value heatmap |
| `rate_comparison.png` | Conversion rate by platform |
| `correction_comparison.png` | Significant pairs by correction method |
| `cpa_comparison_results.csv` | Full t-test results with effect sizes |
| `fisher_comparison_results.csv` | Full Fisher's test results |
| `cpa_confidence_intervals.csv` | Bootstrap 95% CI per platform |
| `power_analysis_cpa.png` | Power curve for CPA differences |
| `power_analysis_results.csv` | Power grid results |

**What to report:**
- Mean ± 95% CI (central tendency + uncertainty)
- N (eval set size — determines power)
- p-value after correction (Bonferroni or BH-FDR)
- Cohen's d (practical significance, not just statistical)